# FindMaximumSubarray e FindMaxCrossingSubarray (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/FindMaximumSubarray_java.ipynb)

Capitolo 17 — Algoritmi ricorsivi. Notebook eseguibile di `FindMaximumSubarray` e `FindMaxCrossingSubarray`: gli algoritmi in Java, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/17-algoritmi-ricorsivi/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/FindMaximumSubarray_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/FindMaximumSubarray_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `recursiveSum` (camelCase).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## FindMaxCrossingSubarray e FindMaximumSubarray (divide et impera)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: un array e tre indici $l\le mid<r$. *Output*: gli indici $maxleft\le mid<maxright$ di un sottoarray di somma massima fra quelli che attraversano il punto medio, e la sua somma. Scandisce la metà sinistra da $mid$ a $l$ e la destra da $mid+1$ a $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: gli indici $i\le j$ di un sottoarray di somma massima di $a[l..r]$ e la sua somma. Il migliore fra il sottoarray massimo della metà sinistra, della destra e di quelli che attraversano il punto medio (a parità, prima il sinistro, poi il destro). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ chiamate.

Il $-\infty$ dello pseudocodice è il valore più piccolo rappresentabile. Prima la traccia sull'esempio $a=[-2,5,-1,3]$ (come l'esecuzione dell'implementazione C++ del capitolo; risultato $(1,3,7)$, cioè $(2,4,7)$ con indici da 1), poi l'esempio della compravendita di azioni, con le variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: tabella dei risultati delle chiamate sui sottointervalli (per livello dell'albero, come nel capitolo; gli indici sono da 0) e i calcoli dei due `FindMaxCrossingSubarray` del livello 1. Il risultato è $(7,10,43)$, cioè $(8,11,43)$ con indici da 1.

In [2]:
%%writefile MaxSubDC.java
import java.util.*;

public class MaxSubDC {
    static int fmsCalls = 0;   // chiamate di FindMaximumSubarray

    static String tr(int[] t) {
        return "(" + t[0] + ", " + t[1] + ", " + t[2] + ")";
    }

    // Restituisce {maxleft, maxright, somma} del miglior sottoarray che attraversa mid
    static int[] findMaxCrossingSubarray(int[] a, int l, int mid, int r) {
        int leftsum = Integer.MIN_VALUE;        // -infinito
        int sum = 0;
        int maxleft = mid;
        for (int i = mid; i >= l; i--) {        // da mid a l
            sum = sum + a[i];                   // somma di a[i..mid]
            if (sum > leftsum) {
                leftsum = sum;
                maxleft = i;
            }
        }
        int rightsum = Integer.MIN_VALUE;
        sum = 0;
        int maxright = mid + 1;
        for (int j = mid + 1; j <= r; j++) {    // da mid+1 a r
            sum = sum + a[j];                   // somma di a[mid+1..j]
            if (sum > rightsum) {
                rightsum = sum;
                maxright = j;
            }
        }
        return new int[]{maxleft, maxright, leftsum + rightsum};
    }

    // Restituisce {i, j, somma} di un sottoarray di somma massima di a[l..r]
    static int[] findMaximumSubarray(int[] a, int l, int r) {
        fmsCalls++;
        if (l == r) {
            return new int[]{l, r, a[l]};       // caso base: un elemento
        }
        int mid = l + (r - l) / 2;
        int[] left = findMaximumSubarray(a, l, mid);
        int[] right = findMaximumSubarray(a, mid + 1, r);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        if (left[2] >= right[2] && left[2] >= cross[2]) {
            return left;
        }
        if (right[2] >= left[2] && right[2] >= cross[2]) {
            return right;
        }
        return cross;
    }

    // ---- varianti che stampano la traccia ----
    static int[] best(int[] left, int[] right, int[] cross) {
        if (left[2] >= right[2] && left[2] >= cross[2]) return left;
        if (right[2] >= left[2] && right[2] >= cross[2]) return right;
        return cross;
    }

    static int[] fmsTrace(int[] a, int l, int r, int depth) {
        String pad = " ".repeat(2 * depth);
        if (l == r) {
            System.out.printf("%sFindMaximumSubarray(a, %d, %d) = (%d, %d, %d)  caso base%n", pad, l, r, l, r, a[l]);
            return new int[]{l, r, a[l]};
        }
        int mid = l + (r - l) / 2;
        System.out.printf("%sFindMaximumSubarray(a, %d, %d): l != r, mid = %d%n", pad, l, r, mid);
        int[] left = fmsTrace(a, l, mid, depth + 1);
        int[] right = fmsTrace(a, mid + 1, r, depth + 1);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        System.out.printf("%s  FindMaxCrossingSubarray(a, %d, %d, %d) = %s%n", pad, l, mid, r, tr(cross));
        int[] ret = best(left, right, cross);
        System.out.printf("%s  left.sum=%d, right.sum=%d, cross.sum=%d => restituisce %s%n", pad, left[2], right[2], cross[2], tr(ret));
        return ret;
    }

    // ogni riga: {l, r, mid} + left + right + cross + ret
    static int[] fmsRows(int[] a, int l, int r, ArrayList<int[][]> rows) {
        if (l == r) {
            return new int[]{l, r, a[l]};
        }
        int mid = l + (r - l) / 2;
        int[] left = fmsRows(a, l, mid, rows);
        int[] right = fmsRows(a, mid + 1, r, rows);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        int[] ret = best(left, right, cross);
        rows.add(new int[][]{{l, r, mid}, left, right, cross, ret});
        return ret;
    }

    static void crossingTrace(int[] a, int l, int mid, int r) {
        System.out.printf("FindMaxCrossingSubarray(a, %d, %d, %d)%n", l, mid, r);
        System.out.printf("%3s %5s %5s %8s %8s%n", "i", "a[i]", "sum", "leftsum", "maxleft");
        int leftsum = Integer.MIN_VALUE, maxleft = mid, sum = 0;
        for (int i = mid; i >= l; i--) {
            sum = sum + a[i];
            if (sum > leftsum) {
                leftsum = sum;
                maxleft = i;
            }
            System.out.printf("%3d %5d %5d %8d %8d%n", i, a[i], sum, leftsum, maxleft);
        }
        System.out.printf("%3s %5s %5s %8s %8s%n", "j", "a[j]", "sum", "rightsum", "maxright");
        int rightsum = Integer.MIN_VALUE, maxright = mid + 1;
        sum = 0;
        for (int j = mid + 1; j <= r; j++) {
            sum = sum + a[j];
            if (sum > rightsum) {
                rightsum = sum;
                maxright = j;
            }
            System.out.printf("%3d %5d %5d %8d %8d%n", j, a[j], sum, rightsum, maxright);
        }
        System.out.printf("sottoarray che attraversa a[%d..%d], somma %d + %d = %d%n", maxleft, maxright, leftsum, rightsum, leftsum + rightsum);
    }

    public static void main(String[] args) {
        int[] a = {-2, 5, -1, 3};
        System.out.println("a = " + Arrays.toString(a));
        int[] res = fmsTrace(a, 0, 3, 0);
        fmsCalls = 0;
        findMaximumSubarray(a, 0, 3);
        System.out.printf("risultato %s   (%d, %d, %d con indici da 1)   chiamate=%d (2n-1=%d)%n", tr(res), res[0] + 1, res[1] + 1, res[2], fmsCalls, 2 * 4 - 1);

        System.out.println();
        int[] b = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        int n = 16;
        System.out.println("a = " + Arrays.toString(b));
        fmsCalls = 0;
        res = findMaximumSubarray(b, 0, n - 1);
        System.out.printf("FindMaximumSubarray(a, 0, %d) = %s   (%d, %d, %d con indici da 1)   chiamate=%d (2n-1=%d)%n", n - 1, tr(res), res[0] + 1, res[1] + 1, res[2], fmsCalls, 2 * n - 1);
        ArrayList<int[][]> rows = new ArrayList<>();
        fmsRows(b, 0, n - 1, rows);
        rows.sort((x, y) -> (x[0][1] - x[0][0]) != (y[0][1] - y[0][0]) ? (y[0][1] - y[0][0]) - (x[0][1] - x[0][0]) : x[0][0] - y[0][0]);
        System.out.printf("%-9s %3s  %-13s %-13s %-13s restituito%n", "call", "mid", "left", "right", "crossing");
        for (int[][] w : rows) {
            String call = "[" + w[0][0] + ".." + w[0][1] + "]";
            System.out.printf("%-9s %3d  %-13s %-13s %-13s %s%n", call, w[0][2], tr(w[1]), tr(w[2]), tr(w[3]), tr(w[4]));
        }
        System.out.println();
        crossingTrace(b, 0, 3, 7);
        System.out.println();
        crossingTrace(b, 8, 11, 15);
    }
}

Writing MaxSubDC.java


In [3]:
!javac MaxSubDC.java && java MaxSubDC

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  caso base
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  caso base
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => restituisce (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  caso base
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  caso base
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => restituisce (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => restituisce (1, 3, 7)
risultato (1, 3, 7)   (2, 4, 7 con indici da 1)   chiamate=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 con indici da 1)   chiamate=31 (2n-1=31)
call      mid  left 

**Prova tu.** Cambia l'array: il sottoarray massimo può avere un solo elemento o attraversare il punto medio; le chiamate sono sempre $2n-1$.

In [4]:
%%writefile ProvaMaxSubDC.java
import java.util.*;

public class ProvaMaxSubDC {
    public static void main(String[] args) {
        // Cambia l'array: quanti elementi ha il sottoarray massimo? Le chiamate sono sempre 2n-1.
        int[] b = {4, -1, 2, 1, -5, 4, -7, 3, 3};
        MaxSubDC.fmsCalls = 0;
        int[] t = MaxSubDC.findMaximumSubarray(b, 0, b.length - 1);
        System.out.println(MaxSubDC.tr(t) + " " + MaxSubDC.fmsCalls);
    }
}

Writing ProvaMaxSubDC.java


In [5]:
!javac ProvaMaxSubDC.java && java ProvaMaxSubDC

(0, 3, 6) 17


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).